In [11]:
# ============================================
# Чекпойнт 7: Единый ноутбук для всех вызовов
# ============================================

import subprocess
import sys
import os
import mlflow
import pandas as pd
import numpy as np
from pathlib import Path
from mlflow.tracking import MlflowClient
from pathlib import Path

# Переходим в нужную директорию
os.chdir(r"C:\Users\Lenovo\YandexDisk\Учеба\ВШЭ ИИ\YearProject\7")
print(f" Рабочая директория: {os.getcwd()}")

 Рабочая директория: C:\Users\Lenovo\YandexDisk\Учеба\ВШЭ ИИ\YearProject\7


In [2]:
# ============================================
# 1. ПРОВЕРКА MLflow СЕРВЕРА
# ============================================

print("\n" + "="*80)
print("1. ПРОВЕРКА MLflow СЕРВЕРА")
print("="*80)

import requests

try:
    response = requests.get("http://127.0.0.1:5000", timeout=5)
    print(" MLflow сервер запущен: http://127.0.0.1:5000")
except:
    print(" MLflow сервер НЕ ЗАПУЩЕН!")
    print("   Запустите в отдельном терминале:")
    print("   python -m mlflow ui --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000")


1. ПРОВЕРКА MLflow СЕРВЕРА
 MLflow сервер запущен: http://127.0.0.1:5000


In [ ]:
# ============================================
# 2. РЕГИСТРАЦИЯ МОДЕЛИ
# ============================================

mlflow.set_tracking_uri("http://127.0.0.1:5000")
client = MlflowClient()

models = client.search_registered_models()
model_exists = any(m.name == "okpd2_classifier" for m in models)

if not model_exists:
    print("Модель не зарегистрирована. Регистрируем...")
    result = subprocess.run([sys.executable, "register_model.py"], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("Ошибки:", result.stderr)
else:
    print("Модель уже зарегистрирована")

Модель уже зарегистрирована


In [4]:
# ============================================
# 3. ДОБАВЛЕНИЕ ТЕГА PRD
# ============================================

result = subprocess.run(
    [sys.executable, "add_prd_tag.py"],
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace'
)
print(result.stdout)
if result.stderr:
    print("Ошибки:", result.stderr)

 Тег PRD добавлен модели okpd2_classifier version 1



In [6]:
# ============================================
# 4. ПОДГОТОВКА ДАННЫХ ДЛЯ АНАЛИЗА (если ещё нет)
# ============================================

data_dir = Path("./data_for_error_analysis")
if not data_dir.exists():
    print("Подготовка данных...")
    result = subprocess.run(
        [sys.executable, "prepare_data_for_error_analysis.py"],
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace'
    )
    print(result.stdout)
    if result.stderr:
        print("Ошибки:", result.stderr)
else:
    print(" Данные уже подготовлены")

 Данные уже подготовлены


In [7]:
# ============================================
# 5. АНАЛИЗ ОШИБОК (ERROR ANALYSIS)
# ============================================

print("Запуск run_error_analysis.py...")
result = subprocess.run(
    [sys.executable, "run_error_analysis.py"],
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace'
)
print(result.stdout)
if result.stderr and "WARNING" not in result.stderr:
    print("Ошибки:", result.stderr)

print("\nАнализ ошибок завершён")

Запуск run_error_analysis.py...
1. Загрузка модели из MLflow...
    Модель загружена

2. Загрузка тестовых данных...
   X_test_pad shape: (29987, 1500)
   y_test shape: (29987, 84)
   Классов: 84
   Текстов загружено: 29987

3. Предсказание...

  1/938 ━━━━━━━━━━━━━━━━━━━━ 41:03 3s/step
  2/938 ━━━━━━━━━━━━━━━━━━━━ 18:07 1s/step
  3/938 ━━━━━━━━━━━━━━━━━━━━ 18:45 1s/step
  4/938 ━━━━━━━━━━━━━━━━━━━━ 18:54 1s/step
  5/938 ━━━━━━━━━━━━━━━━━━━━ 19:16 1s/step
  6/938 ━━━━━━━━━━━━━━━━━━━━ 19:52 1s/step
  7/938 ━━━━━━━━━━━━━━━━━━━━ 20:31 1s/step
  8/938 ━━━━━━━━━━━━━━━━━━━━ 21:19 1s/step
  9/938 ━━━━━━━━━━━━━━━━━━━━ 21:38 1s/step
 10/938 ━━━━━━━━━━━━━━━━━━━━ 21:15 1s/step
 11/938 ━━━━━━━━━━━━━━━━━━━━ 21:00 1s/step
 12/938 ━━━━━━━━━━━━━━━━━━━━ 21:05 1s/step
 13/938 ━━━━━━━━━━━━━━━━━━━━ 21:11 1s/step
 14/938 ━━━━━━━━━━━━━━━━━━━━ 21:11 1s/step
 15/938 ━━━━━━━━━━━━━━━━━━━━ 21:22 1s/step
 16/938 ━━━━━━━━━━━━━━━━━━━━ 21:43 1s/step
 17/938 ━━━━━━━━━━━━━━━━━━━━ 21:57 1s/step
 18/938 ━━━━━━━━━━━━━━━━

In [8]:
# ============================================
# 6. СРАВНЕНИЕ С BASELINE
# ============================================

print("Запуск save_baseline_comparison.py...")
result = subprocess.run(
    [sys.executable, "save_baseline_comparison.py"],
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace'
)
print(result.stdout)
if result.stderr and "WARNING" not in result.stderr:
    print("Ошибки:", result.stderr)

print("\nСравнение с baseline завершено")

Запуск save_baseline_comparison.py...
СРАВНЕНИЕ С BASELINE (на основе результатов чекпойнта 6)

 Метрики Logistic Regression (baseline):
   Accuracy: 0.6771
   F1-macro: 0.5247
   F1-micro: 0.7662
   Hamming Loss: 0.0053

 Метрики Bi-GRU (наша PRD модель):
   Accuracy: 0.7764
   F1-macro: 0.5080
   F1-micro: 0.8177
   Hamming Loss: 0.0043

СРАВНИТЕЛЬНАЯ ТАБЛИЦА
              model  accuracy  f1_macro  f1_micro  hamming_loss
Logistic Regression    0.6771    0.5247    0.7662        0.0053
       Bi-GRU (PRD)    0.7764    0.5080    0.8177        0.0043

 УЛУЧШЕНИЕ Bi-GRU ОТНОСИТЕЛЬНО BASELINE
   Accuracy:    0.6771 → 0.7764  (+9.93%)
   F1-macro:    0.5247 → 0.5080  (-1.67%)
   F1-micro:    0.7662 → 0.8177  (+5.15%)
   Hamming Loss: 0.0053 → 0.0043  (-0.10%)

 ИНТЕРПРЕТАЦИЯ:
   • Bi-GRU лучше по общей точности (Accuracy, F1-micro, Hamming Loss)
   • Logistic Regression лучше работает с редкими классами (F1-macro выше)
   → Bi-GRU выбран как PRD из-за более высокой общей точности, 
     чт

In [9]:
# ============================================
# 7. ROBUSTNESS ТЕСТЫ
# ============================================

print("Запуск robustness_test.py...")
result = subprocess.run(
    [sys.executable, "robustness_test.py"],
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace'
)
print(result.stdout)
if result.stderr and "WARNING" not in result.stderr:
    print("Ошибки:", result.stderr)

print("\nRobustness тесты завершены")

Запуск robustness_test.py (это может занять 10-15 минут)...
ROBUSTNESS ТЕСТЫ

1. Загрузка модели и данных...
   Тестовых примеров: 29987
   Классов: 84

2. Базовые предсказания (без изменений)...

  1/938 ━━━━━━━━━━━━━━━━━━━━ 33:18 2s/step
  2/938 ━━━━━━━━━━━━━━━━━━━━ 18:49 1s/step
  3/938 ━━━━━━━━━━━━━━━━━━━━ 18:06 1s/step
  4/938 ━━━━━━━━━━━━━━━━━━━━ 18:02 1s/step
  5/938 ━━━━━━━━━━━━━━━━━━━━ 18:03 1s/step
  6/938 ━━━━━━━━━━━━━━━━━━━━ 18:16 1s/step
  7/938 ━━━━━━━━━━━━━━━━━━━━ 18:22 1s/step
  8/938 ━━━━━━━━━━━━━━━━━━━━ 18:41 1s/step
  9/938 ━━━━━━━━━━━━━━━━━━━━ 19:01 1s/step
 10/938 ━━━━━━━━━━━━━━━━━━━━ 19:11 1s/step
 11/938 ━━━━━━━━━━━━━━━━━━━━ 19:04 1s/step
 12/938 ━━━━━━━━━━━━━━━━━━━━ 18:59 1s/step
 13/938 ━━━━━━━━━━━━━━━━━━━━ 18:52 1s/step
 14/938 ━━━━━━━━━━━━━━━━━━━━ 18:51 1s/step
 15/938 ━━━━━━━━━━━━━━━━━━━━ 19:00 1s/step
 16/938 ━━━━━━━━━━━━━━━━━━━━ 19:02 1s/step
 17/938 ━━━━━━━━━━━━━━━━━━━━ 19:04 1s/step
 18/938 ━━━━━━━━━━━━━━━━━━━━ 19:15 1s/step
 19/938 ━━━━━━━━━━━━━━━━━━━━ 

In [12]:
# ============================================
# 8. ПРОВЕРКА РЕЗУЛЬТАТОВ
# ============================================

# Проверяем наличие файлов
files_to_check = [
    'baseline_comparison.csv',
    'error_analysis_full.csv',
    'error_analysis_top20.csv',
    'error_analysis_stats.csv',
    'robustness_test_results.csv'
]

print("\n Созданные файлы:")
for f in files_to_check:
    if Path(f).exists():
        size = Path(f).stat().st_size
        print(f"    {f} ({size} bytes)")
    else:
        print(f"    {f} (НЕ НАЙДЕН)")

# Показываем сравнение с baseline
if Path('baseline_comparison.csv').exists():
    print("\n Сравнение с baseline:")
    df = pd.read_csv('baseline_comparison.csv')
    print(df.to_string(index=False))


 Созданные файлы:
    baseline_comparison.csv (137 bytes)
    error_analysis_full.csv (90070 bytes)
    error_analysis_top20.csv (14378 bytes)
    error_analysis_stats.csv (59 bytes)
    robustness_test_results.csv (334 bytes)

 Сравнение с baseline:
              model  accuracy  f1_macro  f1_micro  hamming_loss
Logistic Regression    0.6771    0.5247    0.7662        0.0053
       Bi-GRU (PRD)    0.7764    0.5080    0.8177        0.0043


In [13]:
# ============================================
# 9. ПРОВЕРКА ЗАГРУЗКИ МОДЕЛИ ПО ТЕГУ PRD
# ============================================

try:
    model = mlflow.pyfunc.load_model("models:/okpd2_classifier@PRD")
    print("Модель успешно загружена по тегу PRD")
    
    # Тестовое предсказание
    dummy_input = np.zeros((1, 1500))
    pred = model.predict(dummy_input)
    print(f"Тестовое предсказание выполнено, форма выхода: {pred.shape}")
    
except Exception as e:
    print(f"Ошибка загрузки: {e}")
    print("Используйте models:/okpd2_classifier/1")

Модель успешно загружена по тегу PRD
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Тестовое предсказание выполнено, форма выхода: (1, 84)


# Чекпойнт 7: Наблюдаемость и анализ ошибок

## 1. Выбор лучшей модели (PRD)

Выбрана модель: Bi-GRU (Bidirectional GRU)

Обоснование выбора:

| Критерий | Значение | Почему это важно |
|----------|----------|------------------|
| F1-micro | 0.8177 | Высокая общая точность по всем предсказаниям |
| Subset Accuracy | 0.7764 | Точное совпадение всех меток для документа |
| Hamming Loss | 0.0043 | Наименьшая доля ошибок на бинарных метках |
| Учёт последовательности | Да | Bi-GRU учитывает порядок слов, в отличие от MLP |

Почему не MLP: MLP (Wider MLP) показывает лучший F1-macro (0.5608) - то есть лучше работает с редкими классами. Однако для задачи классификации ОКПД2 важнее общая точность (F1-micro), поэтому выбран Bi-GRU.

## 2. Метрики финальной модели (Bi-GRU)

| Метрика | Значение | Описание |
|---------|----------|----------|
| Accuracy | 0.7764 | Доля полностью верных предсказаний |
| F1-micro | 0.8177 | Среднее по всем предсказаниям (взвешенное) |
| F1-macro | 0.5080 | Среднее по классам (каждый класс имеет равный вес) |
| Hamming Loss | 0.0043 | Доля неправильных бинарных предсказаний |


## 3. Сравнение с baseline (Logistic Regression)

| Метрика | Baseline (LogReg) | Bi-GRU (PRD) | Улучшение |
|---------|-------------------|--------------|-----------|
| Accuracy | 0.6771 | 0.7764 | +9.93% |
| F1-macro | 0.5247 | 0.5080 | -1.67% |
| F1-micro | 0.7662 | 0.8177 | +5.15% |
| Hamming Loss | 0.0053 | 0.0043 | -18.9% |

Интерпретация:

- Bi-GRU превосходит Logistic Regression по общей точности (Accuracy, F1-micro, Hamming Loss)
- Logistic Regression лучше работает с редкими классами (F1-macro выше), так как линейная модель менее склонна к переобучению на частых паттернах
- Bi-GRU выбран как PRD из-за более высокой общей точности, что критично для задачи классификации ОКПД2


## 4. Анализ ошибок

### 4.1. Статистика (на 500 тестовых примерах)

| Показатель | Значение |
|------------|----------|
| Всего примеров | 500 |
| Ошибок | 125 (25.0%) |
| Правильных | 375 (75.0%) |

### 4.2. Типы ошибок

| Тип | Количество | Процент от ошибок | Пояснение |
|-----|------------|-------------------|-----------|
| Только пропуск (FN) | 19 | 15.2% | Модель не предсказала правильный код |
| Только лишнее (FP) | 2 | 1.6% | Модель предсказала лишний код |
| И пропуск, и лишнее | 56 | 44.8% | Модель перепутала коды |
| Низкая уверенность (менее 0.6) | 48 | 38.4% | Модель не уверена в предсказании |

### 4.3. Самые проблемные классы (коды ОКПД2)

| Код | Пропущен | Лишний | Всего | Описание (по ОКПД2) |
|-----|----------|--------|-------|---------------------|
| 32 | 16 | 9 | 25 | Работы монтажные |
| 43 | 10 | 7 | 17 | Работы строительные специализированные |
| 26 | 5 | 6 | 11 | Изделия неметаллические |
| 21 | 5 | 7 | 12 | Мебель |
| 20 | 12 | 0 | 12 | Химические продукты |

### 4.4. Примеры ошибок

Ошибка 1: Семантическая близость классов

Текст: "приобретение жилого помещения (благоустроенной квартиры)"

| Параметр | Значение |
|----------|----------|
| Верные коды | 68 (операции с недвижимостью) |
| Предсказанные | 43 (строительные работы) |

Причина: Модель спутала "приобретение" (код 68) со "строительством" (код 43). Классы семантически близки и часто встречаются вместе.

Ошибка 2: Редкий класс

Текст: "услуги по аттестации информационной системы персональных данных"

| Параметр | Значение |
|----------|----------|
| Верные коды | 74 (услуги в области IT) |
| Предсказанные | (ничего) |

Причина: Редкий класс (74). Модель не уверена (уверенность 0.163) и не предсказала ничего.

Ошибка 3: Неоднозначная формулировка

Текст: "поставка специальной одежды"

| Параметр | Значение |
|----------|----------|
| Верные коды | 14, 20, 15 (три кода) |
| Предсказанные | только 14 |

Причина: "Специальная одежда" относится к разным категориям ОКПД2. Модель предсказала только один из трёх верных кодов.

### 4.5. Причины ошибок и возможность корректировки

| Причина | Описание | Корректировка возможна? | Почему? |
|---------|----------|------------------------|---------|
| Семантическая близость | Коды 32 и 43 имеют пересекающиеся словари | Нет | Требуется онтология ОКПД2 (ручной труд экспертов) |
| Редкие классы | Коды 68 и 74 встречаются в 2-3 раза реже | Частично | Нужно в 5-10 раз больше размеченных данных |
| Неоднозначные формулировки | "специальная одежда" относится к нескольким кодам | Нет | Требуется экспертный консенсус |
| Широкие категории | "расходные материалы" - слишком общее понятие | Нет | Проблема определения границ классов |
| Низкая уверенность | Модель не уверена в предсказании | Частично | Добавление контекстных признаков |


## 5. Robustness тесты (устойчивость модели)

### Результаты

| Тест | Accuracy | Падение | Относительное падение | Вывод |
|------|----------|---------|----------------------|-------|
| Базовый | 0.7730 | - | - | Базовое качество |
| Шум (5% токенов) | 0.7462 | 0.0268 | 3.5% | Умеренная устойчивость |
| Удаление каждого 10-го токена | 0.5463 | 0.2267 | 29.3% | Сильная чувствительность |
| Обрезание до 50% длины | 0.7730 | 0.0000 | 0.0% | Отличная устойчивость |
| Синонимические замены | 0.7589 | 0.0141 | 1.8% | Хорошая устойчивость |

### Выводы по устойчивости

| Аспект | Вывод | Рекомендация |
|--------|-------|--------------|
| Шум | Модель умеренно устойчива к опечаткам | Можно добавить аугментацию данных |
| Удаление токенов | Критическая уязвимость | Необходим целостный входной текст |
| Обрезание текста | Ключевая информация - в начале текста | Можно безопасно обрезать до 1500 токенов |
| Синонимы | Модель хорошо обобщает | Уже хорошо, можно усилить через WordNet |

## 6. Результаты экспериментов с DL-моделями

### 6.1. Сводная таблица всех DL-моделей

| Модель | Архитектура | F1-macro | F1-micro | Subset Acc | Hamming Loss | Время (мин) | Параметры (M) |
|--------|-------------|----------|----------|------------|--------------|-------------|---------------|
| Bi-GRU (RNN) | Bi-GRU (2 слоя) | 0.5080 | 0.8177 | 0.7764 | 0.0043 | 84.0 | 26.05 |
| Wider MLP | 1024→512 | 0.5608 | 0.7800 | 0.7070 | 0.0050 | 5.5 | 8.77 |
| Deep MLP | 512→256 | 0.5508 | 0.7857 | 0.7193 | 0.0049 | 5.0 | 4.25 |
| Shallow MLP | 256 | 0.5413 | 0.7766 | 0.6952 | 0.0050 | 3.9 | 2.07 |
| Heavy Dropout MLP | 512→256→128 | 0.3662 | 0.7576 | 0.6749 | 0.0053 | 5.1 | 4.27 |

### 6.2. Почему Bi-GRU лучше MLP

| Критерий | Bi-GRU | MLP |
|----------|--------|-----|
| Учет порядка слов | Да | Нет |
| Контекстная зависимость | Да | Нет |
| Обработка редких классов | Хуже | Лучше |
| Время обучения | Долго (84 мин) | Быстро (4-6 мин) |
| Размер модели | Большой (26M) | Компактный (2-9M) |

### 6.3. Сравнение MLP архитектур

| Архитектура | F1-macro | Вывод |
|-------------|----------|-------|
| Wider MLP | 0.5608 | Лучший F1-macro, но много параметров |
| Deep MLP | 0.5508 | Оптимальный баланс |
| Shallow MLP | 0.5413 | Быстрый прототип |
| Heavy Dropout | 0.3662 | Избыточная регуляризация |

## 7. Общая таблица сравнения всех подходов

| Категория | Модель | F1-macro | F1-micro | Subset Acc | Hamming Loss | Время | Рекомендация |
|-----------|--------|----------|----------|------------|--------------|-------|--------------|
| Baseline | Случайное угадывание | 0.022 | 0.025 | 0.000 | 0.500 | - | Не использовать |
| ML | Logistic Regression | 0.525 | 0.766 | 0.677 | 0.0053 | 3.6 мин | Быстрый старт |
| ML | Linear SVM | 0.548 | 0.756 | 0.680 | 0.0058 | 31.8 мин | Альтернатива LR |
| DL (MLP) | Shallow MLP | 0.541 | 0.777 | 0.695 | 0.0050 | 3.9 мин | Прототип |
| DL (MLP) | Deep MLP | 0.551 | 0.786 | 0.719 | 0.0049 | 5.0 мин | Production (баланс) |
| DL (MLP) | Wider MLP | 0.561 | 0.780 | 0.707 | 0.0050 | 5.5 мин | F1-macro лидер |
| DL (RNN) | Bi-GRU | 0.508 | 0.818 | 0.776 | 0.0043 | 84 мин | F1-micro лидер |

## 8. Развёрнутые выводы по результатам наблюдений

### 8.1. Выводы по качеству моделей

На основе проведённых экспериментов можно сделать следующие выводы о качестве различных типов моделей:

1. Bi-GRU демонстрирует наилучшие показатели по метрикам общей точности: F1-micro (0.8177) и Subset Accuracy (0.7764). Это подтверждает гипотезу о том, что учёт последовательной структуры текста критически важен для задачи классификации контрактных документов. Модель эффективно улавливает контекстные зависимости, например, различие между "исполнитель несёт ответственность" и "исполнитель не несёт ответственность".

2. Wider MLP показывает лучший F1-macro (0.5608), что означает более высокую точность на редких классах. Это объясняется тем, что MLP с большим количеством нейронов (1024→512) способен создавать более детализированные представления признаков, что полезно для малочисленных категорий. Однако платой за это является двукратное увеличение числа параметров по сравнению с Deep MLP (8.77M против 4.25M).

3. Deep MLP представляет собой оптимальный баланс между качеством, скоростью обучения (5.0 мин) и компактностью модели (4.25M параметров). По метрике F1-micro (0.7857) он отстаёт от Bi-GRU всего на 3.2 процентных пункта, но обучается в 16 раз быстрее.

4. Shallow MLP с одним скрытым слоем (256 нейронов) является хорошим выбором для быстрого прототипирования. При минимальном времени обучения (3.9 мин) он показывает достойное качество (F1-micro 0.7766).

5. Heavy Dropout MLP с тремя слоями и повышенной регуляризацией (dropout 0.6) показал наихудшие результаты (F1-macro 0.3662). Это свидетельствует о том, что для Bag-of-Words представления избыточная глубина сети и высокая регуляризация приводят к потере информации на ранних этапах обучения.

### 8.2. Выводы по сравнению DL и ML подходов

Сравнение глубокого обучения с классическими методами машинного обучения позволяет сделать следующие наблюдения:

1. Нейросетевые подходы (MLP и RNN) превосходят классические методы (Logistic Regression, SVM) по всем ключевым метрикам, кроме F1-macro. Это объясняется способностью нейросетей автоматически извлекать иерархические признаки из данных, в то время как линейные модели ограничены линейными разделяющими поверхностями.

2. Logistic Regression остаётся лучшим выбором для быстрого прототипирования и задач, где важна интерпретируемость. При времени обучения 3.6 минуты она демонстрирует F1-micro 0.7662, что лишь на 5 процентных пунктов ниже лучшего результата Bi-GRU.

3. Random Forest показал наихудшие результаты среди всех методов (F1-micro 0.5875, F1-macro 0.3156). Это может быть связано с тем, что для многоклассовой multi-label задачи с 84 классами ансамбль деревьев решений не способен эффективно использовать разреженное BoW-представление.

4. Linear SVM, несмотря на высокое время обучения (31.8 мин), не даёт значительного преимущества перед Logistic Regression, имея близкие значения метрик (F1-micro 0.7555 против 0.7662).

### 8.3. Выводы по анализу ошибок

Анализ ошибок модели Bi-GRU выявил следующие закономерности:

1. Наиболее частый тип ошибки - одновременное наличие пропущенных и лишних кодов (44.8% от всех ошибок). Это свидетельствует о том, что модель часто путает семантически близкие классы, например, код 32 (монтажные работы) и код 43 (строительные работы). Эта проблема не может быть решена простым увеличением объёма данных, так как она связана с фундаментальной неоднозначностью классификации.

2. Высокий процент ошибок с низкой уверенностью модели (38.4%) указывает на то, что модель часто оказывается в ситуации неопределённости. Это характерно для пограничных случаев, когда текст содержит признаки нескольких классов одновременно или когда формулировки не соответствуют типичным паттернам обучающей выборки.

3. Редкие классы (68, 74) систематически пропускаются моделью. Для улучшения ситуации потребовалось бы увеличение обучающей выборки для этих классов в 5-10 раз, что на практике трудно реализуемо из-за ограниченности данных и высокой стоимости разметки.

4. Наличие ошибок с неоднозначными формулировками (например, "специальная одежда" относится к трём разным кодам) указывает на необходимость более глубокого семантического анализа, выходящего за рамки текущего подхода.

### 8.4. Выводы по устойчивости модели

Тестирование устойчивости модели к различным типам возмущений входных данных дало следующие результаты:

1. Модель демонстрирует высокую устойчивость к обрезанию текста (падение качества 0.0%). Это важное наблюдение: вся значимая информация для классификации ОКПД2 содержится в первых 1500 токенах (примерно в первой половине текста). Это позволяет оптимизировать модель, уменьшив максимальную длину последовательности и сократив время инференса.

2. Устойчивость к синонимическим заменам (падение всего 1.8%) свидетельствует о том, что модель успешно обобщает и не "заучивает" конкретные слова, а улавливает семантические паттерны.

3. Умеренная устойчивость к шуму в токенах (падение 3.5%) показывает, что модель справляется с небольшими искажениями, характерными для реальных данных (опечатки, незначительные ошибки распознавания).

4. Критическая уязвимость обнаружена при удалении каждого 10-го токена (падение 29.3%). Это объясняется тем, что Bi-GRU сильно зависит от целостности последовательности. Регулярное удаление токенов разрушает грамматическую структуру и контекстные связи, что делает текст практически нечитаемым для модели.

### 8.5. Итоговые рекомендации

На основе всех проведённых наблюдений сформулированы следующие рекомендации по выбору модели для различных сценариев использования:

| Сценарий | Рекомендуемая модель | Обоснование |
|----------|---------------------|-------------|
| Максимальная общая точность | Bi-GRU | Лучший F1-micro (0.818) и Subset Accuracy (77.6%), учёт структуры текста |
| Максимальная точность на редких классах | Wider MLP | Лучший F1-macro (0.561), хорошо работает с малочисленными категориями |
| Production-среда (баланс) | Deep MLP | Хорошее качество (F1-micro 0.786), быстрое обучение (5 мин), компактный |
| Быстрый прототип | Logistic Regression | 3.6 минуты обучения, интерпретируемость, достойное качество |

## 9. Итоговые артефакты

| Файл | Описание |
|------|----------|
| bigru_prd.keras | Финальная модель Bi-GRU |
| bigru_tokenizer.pkl | Токенизатор для текстов |
| bigru_config.json | Гиперпараметры модели |
| baseline_comparison.csv | Сравнение с Logistic Regression |
| error_analysis_full.csv | Полный анализ ошибок (125 записей) |
| error_analysis_top20.csv | 20 примеров ошибок для отчёта |
| error_analysis_stats.csv | Статистика по типам ошибок |
| robustness_test_results.csv | Результаты тестов устойчивости |


## 10. Доступ к MLflow

MLflow UI: http://127.0.0.1:5000
Model URI: models:/okpd2_classifier@PRD
Experiment: okpd2_classification
